# OCR every JPG at eight rotations

This notebook reads every `.jpg`/`.jpeg` in `dataset_images`, rotates each page counter-clockwise by `0, 45, 90, ..., 315` degrees, runs open-source OCR on every version, and creates a pandas DataFrame with one row per image and eight text columns.

## Short model research (checked 2026-08-09)

| Option | Strengths | Limitation for this task |
|---|---|---|
| Tesseract | Mature, lightweight CPU OCR; many languages | Requires a separate native executable; its orientation/script detection is mainly page-layout based and is not the strongest choice for arbitrary 45-degree trials or embedded scene/ID text |
| EasyOCR | Simple PyTorch API; can retry detected boxes at 90/180/270 degrees | Its built-in `rotation_info` is box-level and documents only right-angle retries; it does not replace the requested whole-page 45-degree sweep |
| docTR | Modern detection + recognition pipeline aimed at documents | Good alternative, but adds no clear advantage over the current PaddleOCR pipeline for these dense printed pages and requires choosing/configuring detector and recognizer pairs |
| **PaddleOCR PP-OCRv6 medium** | End-to-end page text detection + recognition, current open-source release, strong small-text/document support, one model for 50 languages | Heavier than Tesseract/EasyOCR; eight passes per page are intentionally expensive |

## Selection

**Selected: PaddleOCR 3.7 with PP-OCRv6 medium.** The samples are dense 1448×2048 English document pages and one page contains an embedded ID photograph, so a full detector-recognizer pipeline is a better fit than a recognition-only transformer. PaddleOCR's current documentation reports that PP-OCRv6 medium improves detection by 4.6% and recognition by 5.1% over PP-OCRv5_server on its internal multi-scenario benchmark, while using 34.5M parameters. Those figures are vendor benchmark results, not a guarantee on this dataset, but the model/task fit is strong.

Automatic document and text-line orientation modules are disabled below. This is deliberate: every requested page rotation must be OCRed independently rather than silently normalized first.

Sources: [PaddleOCR general OCR pipeline](https://www.paddleocr.ai/main/en/version3.x/pipeline_usage/OCR.html), [PP-OCRv6 introduction](https://www.paddleocr.ai/main/en/version3.x/algorithm/PP-OCRv6/PP-OCRv6.html), [PaddleOCR package/release](https://pypi.org/project/paddleocr/), [EasyOCR API](https://www.jaided.ai/easyocr/documentation/), [Tesseract quality/orientation guidance](https://tesseract-ocr.github.io/tessdoc/ImproveQuality.html).

## 1. Install dependencies

The cell below installs the CPU runtime. Restart the kernel once after installation if Jupyter asks. For an NVIDIA GPU, install the PaddlePaddle build matching your CUDA version from the [official installation guide](https://www.paddleocr.ai/main/en/quick_start.html), then set `DEVICE = "gpu"` below. Python 3.10–3.12 is the safest environment range.

In [1]:
%pip install -q "paddlepaddle==3.2.0" -i https://www.paddlepaddle.org.cn/packages/stable/cpu/
%pip install -q "paddleocr==3.7.0" "pandas>=2.0" "Pillow>=10.0" "tqdm>=4.66"

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.2
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Imports and configuration

Positive PIL angles are counter-clockwise. `expand=True` prevents diagonal rotations from cropping page corners, and a white fill avoids adding black triangles that can confuse text detection.

In [2]:
import json
from pathlib import Path

import numpy as np
import pandas as pd
from PIL import Image, ImageOps
from paddleocr import PaddleOCR
from tqdm.auto import tqdm

DATASET_DIR = Path("dataset_images")
OUTPUT_CSV = Path("ocr_8_angles.csv")
ANGLES = tuple(range(0, 360, 45))
IMAGE_EXTENSIONS = {".jpg", ".jpeg"}
DEVICE = "cpu"       # Change to "gpu" after installing a matching GPU runtime.
MAX_IMAGES = None      # Set an integer such as 10 for a quick test run.

assert ANGLES == (0, 45, 90, 135, 180, 225, 270, 315)

C:\Users\user\PycharmProjects\Project.DS\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## 3. Load PP-OCRv6 medium

The model weights download automatically on the first run and are cached. The explicit model names pin the accuracy-oriented medium detector and recognizer.

In [3]:
# ocr = PaddleOCR(
#     ocr_version="PP-OCRv6",
#     lang="en",
#     text_detection_model_name="PP-OCRv6_small_det",
#     text_recognition_model_name="PP-OCRv6_small_rec",
#     text_recognition_batch_size=8,
#     use_doc_orientation_classify=False,
#     use_doc_unwarping=False,
#     use_textline_orientation=False,
#     device=DEVICE,
# )

In [4]:
ocr = PaddleOCR(
    ocr_version="PP-OCRv6",
    lang="en",
    text_detection_model_name="PP-OCRv6_tiny_det",
    text_recognition_model_name="PP-OCRv6_tiny_rec",
    text_recognition_batch_size=8,
    use_doc_orientation_classify=False,
    use_doc_unwarping=False,
    use_textline_orientation=False,
    device=DEVICE,
)

C:\Users\user\AppData\Local\Temp\ipykernel_13680\626481610.py:1: UserWarning: `lang` and `ocr_version` will be ignored when model names or model directories are not `None`.
  ocr = PaddleOCR(
Creating model: ('PP-OCRv6_tiny_det', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\user\.paddlex\official_models\PP-OCRv6_tiny_det`.
C:\Users\user\PycharmProjects\Project.DS\.venv\Lib\site-packages\paddle\utils\cpp_extension\extension_utils.py:718: UserWarning: No ccache found. Please be aware that recompiling all source files may be required. You can download and install ccache from: https://github.com/ccache/ccache/blob/master/doc/INSTALL.md
  warnings.warn(warning_message)
Creating model: ('PP-OCRv6_tiny_rec', None, None)
Model files already exist. Using cached files. To redownload, please delete the directory manually: `C:\Users\user\.paddlex\official_models\PP-OCRv6_tiny_rec`.


## 4. Rotation and OCR helpers

PaddleOCR returns recognized lines in detection order. They are joined with newlines so each DataFrame cell contains one page-level string.

In [5]:
def rotate_page(image: Image.Image, angle: int) -> Image.Image:
    """Rotate counter-clockwise without cropping and use a white background."""
    image = ImageOps.exif_transpose(image).convert("RGB")
    if angle == 0:
        return image.copy()
    return image.rotate(
        angle,
        resample=Image.Resampling.BICUBIC,
        expand=True,
        fillcolor=(255, 255, 255),
    )


def result_payload(result_item) -> dict:
    """Normalize PaddleOCR/PaddleX result representations to a dictionary."""
    if isinstance(result_item, dict):
        payload = result_item
    else:
        payload = getattr(result_item, "json", None)
        if callable(payload):
            payload = payload()

    if isinstance(payload, str):
        payload = json.loads(payload)
    if not isinstance(payload, dict):
        raise TypeError(f"Unexpected OCR result type: {type(result_item)!r}")

    # PaddleX JSON output commonly wraps the actual OCR fields under 'res'.
    return payload.get("res", payload)


def extract_page_text(image: Image.Image, ocr_engine=ocr) -> str:
    """Run OCR and return all non-empty recognized lines as one string."""
    lines = []
    for result_item in ocr_engine.predict(np.asarray(image)):
        payload = result_payload(result_item)
        lines.extend(
            str(text).strip()
            for text in payload.get("rec_texts", [])
            if str(text).strip()
        )
    return "\n".join(lines)

## 5. Process the dataset

The output schema is `image_name` plus exactly eight text columns: `text_angle_000` through `text_angle_315`. OCR is run 8 × number-of-images times, so start with `MAX_IMAGES = 2` if you want a timing check.

In [6]:
def extract_page_text_fast(image):
    lines = []

    results = ocr.predict(
        np.asarray(image),
        text_det_limit_side_len=1600,
        text_det_limit_type="max",
    )

    for result_item in results:
        payload = result_payload(result_item)

        lines.extend(
            str(text).strip()
            for text in payload.get("rec_texts", [])
            if str(text).strip()
        )

    return "\n".join(lines)


if not DATASET_DIR.is_dir():
    raise FileNotFoundError(f"Directory not found: {DATASET_DIR.resolve()}")

image_paths = sorted(
    path
    for path in DATASET_DIR.iterdir()
    if path.is_file() and path.suffix.lower() in IMAGE_EXTENSIONS
)

if MAX_IMAGES is not None:
    image_paths = image_paths[:MAX_IMAGES]

if not image_paths:
    raise ValueError(f"No JPG images found in {DATASET_DIR.resolve()}")


text_columns = [
    f"text_angle_{angle:03d}"
    for angle in ANGLES
]

rows = []

total_ocr_runs = len(image_paths) * len(ANGLES)

with tqdm(total=total_ocr_runs, desc="OCR passes") as progress:

    for image_path in image_paths:

        row = {"image_name": image_path.name}

        with Image.open(image_path) as source_image:
            source_image.load()

            for angle in ANGLES:

                progress.set_postfix(
                    image=image_path.name[:20],
                    angle=angle
                )

                rotated_image = rotate_page(
                    source_image,
                    angle
                )

                row[f"text_angle_{angle:03d}"] = (
                    extract_page_text_fast(rotated_image)
                )

                progress.update(1)

        rows.append(row)

        # Save a checkpoint after every completed image
        partial_df = pd.DataFrame(
            rows,
            columns=["image_name", *text_columns]
        )

        partial_df.to_csv(
            "ocr_8_angles_partial.csv",
            index=False,
            encoding="utf-8-sig"
        )


documents_df = pd.DataFrame(
    rows,
    columns=["image_name", *text_columns]
)

documents_df

OCR passes: 100%|██████████| 8/8 [01:12<00:00,  9.09s/it, angle=315, image=PDF_Edge_Case_11_pag]


,image_name,text_angle_000,text_angle_045,text_angle_090,text_angle_135,text_angle_180,text_angle_225,text_angle_270,text_angle_315
0,PDF_Edge_Case_11_page_1.jpg,Random document\nresult simple system model ex...,Random document\nresult simple system model ex...,ejduis eqnu Jequnu wopuri e6ed sseooud insei e...,1 ↓0labed\nluewnoop ejdexe sse0oud e!! ueinoop...,lJolabed\nsse0oid epow luewnoop anjeΛ efed eed...,1 Jol abed\nsse0oud epo uaunoop enje ebed ebed...,value example system process page page value d...,Random document\nresult simple system model ex...


## 6. Save and preview

CSV preserves the eight page strings, including embedded newline characters. `documents_df` remains available in memory for further analysis.

In [7]:
documents_df.to_csv(OUTPUT_CSV, index=False, encoding="utf-8-sig")
print(f"Saved {len(documents_df):,} rows to {OUTPUT_CSV.resolve()}")

preview_df = documents_df.copy()
for column in text_columns:
    preview_df[column] = preview_df[column].str.replace("\n", " ", regex=False).str[:160]
preview_df.head()

Saved 1 rows to C:\Users\user\PycharmProjects\Project.DS\ocr_8_angles.csv


,image_name,text_angle_000,text_angle_045,text_angle_090,text_angle_135,text_angle_180,text_angle_225,text_angle_270,text_angle_315
0,PDF_Edge_Case_11_page_1.jpg,Random document result simple system model exa...,Random document result simple system model exa...,ejduis eqnu Jequnu wopuri e6ed sseooud insei e...,1 ↓0labed luewnoop ejdexe sse0oud e!! ueinoop ...,lJolabed sse0oid epow luewnoop anjeΛ efed eed ...,1 Jol abed sse0oud epo uaunoop enje ebed ebed ...,value example system process page page value d...,Random document result simple system model exa...


## Interpretation note

With 45-degree steps, the nearest trial can still be 22.5 degrees away from an arbitrary source angle. This notebook follows the requested eight-angle design exactly; if the real dataset contains continuously varying skew and accuracy is insufficient, estimate the skew angle first or use smaller steps. Also benchmark the output against a manually transcribed sample before production use—OCR confidence and model release claims do not replace dataset-specific character/word error measurement.